# Chemical and elastic phase separation in Julia

I use the website’s dimensionless two-dimensional model, with periodic boundaries, unit mobility and unit gradient coefficient. Composition c is a conserved field. The bulk free-energy density is c²(1-c)². Its derivative contributes 2c(1-c)(1-2c) to chemical potential. The elastic coefficient B(k) is zero for the chemical example. The elastic example uses a homogeneous 2D constitutive model with zero mean stress; it is not a calibrated alloy or a plane-strain reduction of a 3D eigenstrain.

Run each small part and its checks in order. Julia indices start at 1. For Fourier examples install FFTW with `julia -e 'using Pkg; Pkg.add("FFTW")'`. For notebook use, install IJulia and select its kernel. The script is also supplied. Julia and NumPy random sequences differ; compare ensemble statistics, not individual paths.

## 1. Create wave numbers and the initial field

I start with the chemical case. The grid spacing is one dimensionless length unit. I remove the mean of the deterministic perturbation so that the initial mean composition is exactly 0.5 to floating-point accuracy.

In [ ]:
using FFTW, Statistics
n = 64
dt = 0.2
k = 2π .* fftfreq(n)
kx = repeat(reshape(k, n, 1), 1, n)
ky = permutedims(kx)
k2 = kx.^2 .+ ky.^2
noise = [sin(row*12.9898 + column*78.233)*43758.5453 for row in 1:n, column in 1:n]
noise = 0.02 .* (noise .- floor.(noise) .- 0.5)
noise .-= mean(noise)
initial = 0.5 .+ noise
@assert abs(mean(initial) - 0.5) < 1e-12
@assert k2[1, 1] == 0
println("Initial mean: ", mean(initial))

## 2. Write one update and test conservation

The gradient and elastic terms are implicit; the nonlinear bulk derivative is explicit. The zero Fourier mode is unchanged because k² is zero there. I check the mean and finite values after one update. These checks do not prove energy stability.

In [ ]:
chemical_kernel = zeros(size(k2))
bulk_derivative = 2 .* initial .* (1 .- initial) .* (1 .- 2 .* initial)
numerator = fft(initial) .- dt .* k2 .* fft(bulk_derivative)
denominator = 1 .+ dt .* k2 .* (k2 .+ chemical_kernel)
one_step = real.(ifft(numerator ./ denominator))
@assert all(isfinite, one_step)
@assert abs(mean(one_step) - mean(initial)) < 1e-12
println("Mean change: ", mean(one_step) - mean(initial))

## 3. Add the elastic kernel separately

I use the existing tested kernel rather than introducing a different formula in this notebook. C11, C12 and C44 are dimensionless elastic stiffnesses here. The dilatational eigenstrain is dimensionless. I check the isotropic limit, in which the nonzero-mode kernel has no angular dependence. I also check that zero misfit removes the elastic contribution. I symmetrize conjugate Fourier partners for a real field on an even grid.

In [ ]:
include("../phase_field.jl")
angles = range(0, 2π; length=33)
isotropic = kernel(cos.(angles), sin.(angles); c11=4, c12=2, c44=1)
@assert maximum(isotropic) - minimum(isotropic) < 1e-12
@assert all(abs.(kernel(kx, ky; eigenstrain=0)) .< 1e-12)
elastic_kernel = kernel(kx, ky; c11=4, c12=2, c44=3, eigenstrain=0.2)
partner = mod.(-(0:n-1), n) .+ 1
elastic_kernel = 0.5 .* (elastic_kernel .+ elastic_kernel[partner, partner])
@assert all(isfinite, elastic_kernel)
println("Isotropic angular variation: ", maximum(isotropic) - minimum(isotropic))

## 4. Compare short evolutions with the same initial field

I advance both cases from identical fields for 50 steps. This is a short execution and conservation check. The complete program supplies longer runs and free-energy records. Timestep and spatial refinement are separate questions and have not been established by this notebook.

In [ ]:
chemical = copy(initial)
elastic = copy(initial)
for step in 1:50
    chemical_bulk = 2 .* chemical .* (1 .- chemical) .* (1 .- 2 .* chemical)
    chemical_spectrum = fft(chemical) .- dt .* k2 .* fft(chemical_bulk)
    chemical .= real.(ifft(chemical_spectrum ./ (1 .+ dt .* k2 .* (k2 .+ chemical_kernel))))
    elastic_bulk = 2 .* elastic .* (1 .- elastic) .* (1 .- 2 .* elastic)
    elastic_spectrum = fft(elastic) .- dt .* k2 .* fft(elastic_bulk)
    elastic .= real.(ifft(elastic_spectrum ./ (1 .+ dt .* k2 .* (k2 .+ elastic_kernel))))
end
for result in (chemical, elastic)
    @assert all(isfinite, result)
    @assert abs(mean(result) - mean(initial)) < 1e-10
end
@assert sum(abs2, chemical .- elastic) > 0
println("Final time: ", 50 * dt)
println("Field RMS difference: ", sqrt(mean((chemical .- elastic).^2)))